In [ ]:
import sys
print(sys.version)

In [ ]:
pip install opencv-python mediapipe

In [ ]:
import mediapipe as mp
print(mp.__version__)
print(hasattr(mp, "solutions"))

In [ ]:
# ====================================================================
# PROGRAMA: FOTOS SEGUN NUMERO DE DEDOS (UNA MANO)
#
# Descripcion:
# Este programa utiliza la camara web y MediaPipe Hands para detectar
# cuantos dedos tiene levantados el usuario con una sola mano.
# Segun el numero de dedos (0 a 5), se toma automaticamente una foto
# y se guarda en la carpeta 'fotos_capturadas'.
#
# Controles:
# - Muestra los dedos frente a la camara (0 a 5).
# - Al mantener un numero de dedos ~2 segundos, se toma la foto.
# - ESC para salir.
#
# Librerias:
# - OpenCV (cv2): Captura de video y procesamiento de imagenes.
# - MediaPipe (mp): Deteccion de landmarks de la mano.
# - os, time: Manejo de archivos y temporizadores.
# ====================================================================
import cv2
import mediapipe as mp
import os
import time
from datetime import datetime

# --- INICIALIZACION DE MEDIAPIPE ---
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(static_image_mode=False, max_num_hands=1,
                       min_detection_confidence=0.7, min_tracking_confidence=0.7)
mp_draw = mp.solutions.drawing_utils

# --- CREAR CARPETA PARA GUARDAR FOTOS ---
carpeta_fotos = "fotos_capturadas"
if not os.path.exists(carpeta_fotos):
    os.makedirs(carpeta_fotos)

# --- INICIALIZACION DE LA CAMARA ---
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

def contar_dedos(hand_landmarks):
    """
    Cuenta cuantos dedos estan levantados usando los landmarks.
    
    Logica:
    - Pulgar (ID 4): Se compara coordenada x con nudillo (ID 3).
      Para mano derecha: pulgar levantado si x_4 > x_3.
      Para mano izquierda: pulgar levantado si x_4 < x_3.
    - Otros dedos: Se compara punta (ID 8,12,16,20) con nudillo medio
      (ID 6,10,14,18). Si punta_y < nudillo_y, el dedo esta levantado.
    """
    lm = hand_landmarks.landmark
    dedos = []

    # Pulgar (ID 4 vs ID 3)
    if lm[4].x < lm[3].x:
        dedos.append(1)
    else:
        dedos.append(0)

    # Indice (ID 8 vs ID 6)
    if lm[8].y < lm[6].y:
        dedos.append(1)
    else:
        dedos.append(0)

    # Medio (ID 12 vs ID 10)
    if lm[12].y < lm[10].y:
        dedos.append(1)
    else:
        dedos.append(0)

    # Anular (ID 16 vs ID 14)
    if lm[16].y < lm[14].y:
        dedos.append(1)
    else:
        dedos.append(0)

    # Menique (ID 20 vs ID 18)
    if lm[20].y < lm[18].y:
        dedos.append(1)
    else:
        dedos.append(0)

    return sum(dedos)

# --- VARIABLES DE CONTROL ---
num_dedos_actual = 0
num_dedos_anterior = -1
tiempo_inicio_gesto = 0
foto_tomada = False
mensaje = ""
tiempo_mensaje = 0

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    frame = cv2.flip(frame, 1)
    height, width, _ = frame.shape
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(rgb_frame)

    if results.multi_hand_landmarks:
        hand_landmarks = results.multi_hand_landmarks[0]
        mp_draw.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)

        num_dedos_actual = contar_dedos(hand_landmarks)

        # Si el numero de dedos cambio, reiniciar el temporizador
        if num_dedos_actual != num_dedos_anterior:
            tiempo_inicio_gesto = time.time()
            num_dedos_anterior = num_dedos_actual
            foto_tomada = False

        # Si se mantiene el mismo numero ~2 segundos, tomar foto
        if (not foto_tomada and
            time.time() - tiempo_inicio_gesto > 2.0):
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
            nombre_foto = f"{carpeta_fotos}/foto_{num_dedos_actual}_dedos_{timestamp}.jpg"
            cv2.imwrite(nombre_foto, frame)
            foto_tomada = True
            mensaje = f"Foto guardada: {num_dedos_actual} dedos!"
            tiempo_mensaje = time.time()

        # Mostrar el numero de dedos en pantalla
        cv2.putText(frame, f"Dedos: {num_dedos_actual}",
                    (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 3)

        # Barra de progreso para la captura
        if not foto_tomada:
            progreso = min((time.time() - tiempo_inicio_gesto) / 2.0, 1.0)
            ancho_barra = int(200 * progreso)
            cv2.rectangle(frame, (10, 70), (10 + ancho_barra, 85), (0, 255, 255), cv2.FILLED)
            cv2.rectangle(frame, (10, 70), (210, 85), (255, 255, 255), 1)
    else:
        # No hay mano detectada
        num_dedos_anterior = -1
        tiempo_inicio_gesto = 0
        foto_tomada = False

    # Mostrar mensaje temporal
    if mensaje and time.time() - tiempo_mensaje < 2.0:
        cv2.putText(frame, mensaje, (width // 2 - 200, height // 2),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 255), 3)
    else:
        mensaje = ""

    # Instrucciones
    cv2.putText(frame, "Muestra 0-5 dedos ~2s para foto | ESC salir",
                (10, height - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)

    cv2.imshow("Fotos segun dedos", frame)

    if cv2.waitKey(1) & 0xFF == 27 or cv2.getWindowProperty("Fotos segun dedos", cv2.WND_PROP_VISIBLE) < 1:
        break

cap.release()
cv2.destroyAllWindows()

print(f"Fotos guardadas en la carpeta: {carpeta_fotos}/")